In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from pathlib import Path
import re
import numpy as np
import pandas as pd
from itertools import combinations
from math import log2
from scipy.spatial.distance import cosine

# ========= USER SETTINGS =========
ROOT = Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No")
FIGS_OH = ROOT / "figs_OH"
FIGS_FP = ROOT / "figs_FP"
# onlyMat を自動検出（無ければ明示指定に差し替え可）
FEAT_OH = next(iter(sorted(ROOT.glob("features*OH*onlyMat*.csv"))), ROOT / "features_OH_onlyMat.csv")
FEAT_FP = next(iter(sorted(ROOT.glob("features*FP*onlyMat*.csv"))), ROOT / "features_FP_onlyMat.csv")

OUTDIR  = ROOT / "OHFP_varmap"
OUTDIR.mkdir(parents=True, exist_ok=True)

DIMS    = ["top3","cumeig"]
INDICES = ["silhouette","dunn","gap","ch","db","ptbiserial"]

# ハイパラ：材料→フラグメント対応を集計する際の閾値など
MIN_SAMPLES_PER_MATERIAL = 1         # その材料が出現する最小サンプル数（これ未満は除外）
FRAG_PREV_THRESHOLD      = 0.20      # P(fragment=1 | material=1) がこの閾値未満ならノイズとして無視
MIN_EFFECTIVE_FRAGS      = 3         # 上記閾値を超えるフラグメントの最小数（未満は弱いとみなす）

# ========= helpers =========
FN_RE = re.compile(
    r"^ClusterAssign_(?P<mode>top3|cumeig)_(?P<index>silhouette|dunn|gap|ch|db|ptbiserial)_(?P<ds>OH|FP)_(?P<date>\d{8})_(?P<hm>\d{4})\.csv$"
)

def latest_by_mode_index(fig_dir: Path, ds: str):
    latest = {}
    for p in fig_dir.glob("ClusterAssign_*.csv"):
        m = FN_RE.match(p.name)
        if not m or m["ds"] != ds: continue
        key = (m["mode"], m["index"])
        ts  = f"{m['date']}_{m['hm']}"
        if key not in latest or ts > latest[key][0]:
            latest[key] = (ts, p)
    return {k:v[1] for k,v in latest.items()}

def read_var_cluster(path: Path):
    df = pd.read_csv(path)
    cols = {c.lower(): c for c in df.columns}
    vcol = cols.get("variable", df.columns[0])
    ccol = cols.get("cluster",  df.columns[-1])
    s = pd.Series(df[ccol].values, index=df[vcol].astype(str).values, name="cluster")
    try: s = s.astype(int)
    except: pass
    return s

def load_feat(path: Path):
    df = pd.read_csv(path)
    # 1列目を sample_id とみなす
    if df.columns[0].lower() != "sample_id":
        df = df.rename(columns={df.columns[0]:"sample_id"})
    df = df.set_index("sample_id").apply(pd.to_numeric, errors="coerce").fillna(0.0)
    return (df != 0).astype(int)

def entropy(p):
    # p: 1D array-like prob dist (sum=1). 0*log0=0扱い。
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    if len(p)==0: return 0.0
    return -float(np.sum(p * np.log2(p)))

def js_divergence(p, q):
    # Jensen-Shannon divergence (base2)
    p = np.asarray(p, dtype=float); p = p / (p.sum() + 1e-12)
    q = np.asarray(q, dtype=float); q = q / (q.sum() + 1e-12)
    m = 0.5*(p+q)
    def _kl(a,b):
        a = np.clip(a,1e-12,1); b = np.clip(b,1e-12,1)
        return float(np.sum(a * (np.log2(a) - np.log2(b))))
    return 0.5*_kl(p,m) + 0.5*_kl(q,m)

def material_to_fp_distribution(material, Xoh, Xfp, fp_var2clu, threshold=0.2):
    # サンプル集合（その材料が1のサンプル）
    idx = Xoh.index[Xoh[material] == 1]
    if len(idx) == 0:
        return None, None, 0, {}
    # 条件付き出現率 P(fragment=1 | material=1)
    prev = Xfp.loc[idx].mean(axis=0)  # Series: fragment -> prevalence
    # 閾値以上のフラグメントだけを採用
    sel = prev[prev >= threshold]
    if sel.empty:
        return None, None, len(idx), {}
    # FP クラスタごとに“重み”（=出現率の合計）を集計
    df = pd.DataFrame({"frag": sel.index, "w": sel.values})
    df["fp_cluster"] = df["frag"].map(fp_var2clu).astype("Int64")
    df = df.dropna(subset=["fp_cluster"])
    if df.empty:
        return None, None, len(idx), {}
    grp = df.groupby("fp_cluster", as_index=False)["w"].sum()
    # 分布（確率化）
    grp["p"] = grp["w"] / grp["w"].sum()
    dist = dict(zip(grp["fp_cluster"].astype(int), grp["p"]))
    # majority と purity
    maj = int(grp.sort_values("p", ascending=False).iloc[0]["fp_cluster"])
    purity = float(grp["p"].max())
    return maj, purity, len(idx), dist

def run_one(mode, index, FEAT_OH, FEAT_FP, oh_assign, fp_assign):
    Xoh = load_feat(FEAT_OH)
    Xfp = load_feat(FEAT_FP)
    common = Xoh.index.intersection(Xfp.index)
    if len(common)==0:
        print(f"[WARN] No common samples for {mode} × {index}")
        return

    Xoh = Xoh.loc[common]
    Xfp = Xfp.loc[common]

    # この (mode,index) に登場する“材料”（=OH側 ClusterAssign の Variable）
    materials = [m for m in oh_assign.index if m in Xoh.columns]
    # FP 側の “フラグメント” 辞書
    fp_var2clu = fp_assign

    rows_mat = []
    for mat in materials:
        # サンプル数フィルタ
        n_mat = int(Xoh[mat].sum())
        if n_mat < MIN_SAMPLES_PER_MATERIAL:
            continue
        maj, purity, n_s, dist = material_to_fp_distribution(
            mat, Xoh, Xfp, fp_var2clu, threshold=FRAG_PREV_THRESHOLD
        )
        oh_c = int(oh_assign.get(mat, np.nan)) if pd.notna(oh_assign.get(mat, np.nan)) else np.nan
        # エントロピー（分布が広いほど高い）
        if dist:
            pvec = np.array(list(dist.values()), dtype=float)
            H = entropy(pvec / (pvec.sum() + 1e-12))
            k_eff = len(dist)
        else:
            H = np.nan; k_eff = 0
        rows_mat.append({
            "mode": mode, "index": index,
            "material": mat,
            "OH_cluster": oh_c,
            "n_samples_with_material": n_s,
            "FP_major_cluster": maj,
            "FP_purity": purity,
            "FP_entropy": H,
            "FP_k_effective": k_eff,
            "FP_dist": dist
        })

    df_mat = pd.DataFrame(rows_mat)
    out1 = OUTDIR / f"material_to_fpcluster_{mode}_{index}.csv"
    if len(df_mat):
        # dist（辞書）を安全に文字列化
        df_mat = df_mat.assign(FP_dist=df_mat["FP_dist"].apply(lambda d: ";".join(f"{k}:{v:.3f}" for k,v in sorted(d.items())) if isinstance(d, dict) and d else ""))
        df_mat.to_csv(out1, index=False)
        print(f"[SAVED] {out1}")
    else:
        print(f"[INFO] No materials mapped: {mode} × {index}")

    # 同一OHクラスター内の材料ペアの“分岐/一致”評価
    rows_pair = []
    if len(df_mat):
        # dict: material -> prob vector on FP clusters（共通順序で並べる）
        # まず全FPクラスタIDの全集合
        all_fp_clusters = sorted(set(k for d in rows_mat for k in (d["FP_dist"] or {}).keys()))
        def vec_of(dist):
            return np.array([dist.get(k, 0.0) for k in all_fp_clusters], dtype=float)

        for k, grp in df_mat.dropna(subset=["OH_cluster"]).groupby("OH_cluster"):
            mats = grp["material"].tolist()
            for a, b in combinations(mats, 2):
                da = next(d for d in rows_mat if d["material"]==a)["FP_dist"] or {}
                db = next(d for d in rows_mat if d["material"]==b)["FP_dist"] or {}
                va, vb = vec_of(da), vec_of(db)
                # 指標：major一致, purityのmin, cosine類似度, JS divergence
                maj_same = (next(d for d in rows_mat if d["material"]==a)["FP_major_cluster"] ==
                            next(d for d in rows_mat if d["material"]==b)["FP_major_cluster"])
                purity_min = float(min(
                    next(d for d in rows_mat if d["material"]==a)["FP_purity"] or 0.0,
                    next(d for d in rows_mat if d["material"]==b)["FP_purity"] or 0.0,
                ))
                # cosine は「似ているほど1に近い」ので 1-cosine_distance を類似度として扱う
                cos_sim = 1.0 - (cosine(va, vb) if (va.sum()>0 and vb.sum()>0) else 1.0)
                jsd = js_divergence(va, vb) if (va.sum()>0 and vb.sum()>0) else np.nan
                rows_pair.append({
                    "mode": mode, "index": index,
                    "OH_cluster": int(k),
                    "material_A": a,
                    "material_B": b,
                    "FP_major_same": bool(maj_same),
                    "FP_purity_min": purity_min,
                    "FP_cosine_similarity": cos_sim,
                    "FP_JS_divergence": jsd
                })
    df_pair = pd.DataFrame(rows_pair)
    out2 = OUTDIR / f"materialpair_cohesion_{mode}_{index}.csv"
    if len(df_pair):
        df_pair.to_csv(out2, index=False)
        print(f"[SAVED] {out2}")

        # OHクラスター単位の集計： “まとまる”割合（major一致 & ある程度の純度）
        def label_cohesion(r, purity_thr=0.6, cos_thr=0.6, js_thr=0.5):
            # どれか1条件でも満たせば “まとまる” と判定する緩い版
            return bool(
                (r["FP_major_same"] and r["FP_purity_min"] >= purity_thr) or
                (r["FP_cosine_similarity"] >= cos_thr) or
                (pd.notna(r["FP_JS_divergence"]) and r["FP_JS_divergence"] <= js_thr)
            )
        df_pair["cohesive_flag"] = df_pair.apply(label_cohesion, axis=1)
        summary = (df_pair.groupby(["mode","index","OH_cluster"], as_index=False)
                          .agg(n_pairs=("cohesive_flag","size"),
                               n_cohesive=("cohesive_flag","sum")))
        summary["cohesive_ratio"] = summary["n_cohesive"] / summary["n_pairs"].replace(0, np.nan)
        out3 = OUTDIR / f"ohcluster_consistency_{mode}_{index}.csv"
        summary.to_csv(out3, index=False)
        print(f"[SAVED] {out3}")

def main():
    # 最新ファイルで (mode,index) をそろえる
    oh = latest_by_mode_index(FIGS_OH, "OH")
    fp = latest_by_mode_index(FIGS_FP, "FP")
    keys = [(m,i) for m in DIMS for i in INDICES if (m,i) in oh and (m,i) in fp]
    if not keys:
        print("[ERROR] No overlapping (mode,index) between OH and FP.")
        return

    for (mode,index) in keys:
        print(f"\n=== {mode} × {index} ===")
        oh_assign = read_var_cluster(oh[(mode,index)])
        fp_assign = read_var_cluster(fp[(mode,index)])
        run_one(mode, index, FEAT_OH, FEAT_FP, oh_assign, fp_assign)

if __name__ == "__main__":
    main()



=== top3 × silhouette ===
[SAVED] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap/material_to_fpcluster_top3_silhouette.csv
[SAVED] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap/materialpair_cohesion_top3_silhouette.csv
[SAVED] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap/ohcluster_consistency_top3_silhouette.csv

=== top3 × dunn ===
[SAVED] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap/material_to_fpcluster_top3_dunn.csv
[SAVED] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap/materialpair_cohesion_top3_dunn.csv
[SAVED] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap/ohcluster_consistency_top3_dunn.csv

=== top3 × gap ===
[SAVED] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap/material_to_fpcluster_top3_gap.csv
[SAVED] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap/materialpair_cohesion_t

In [2]:
# # -*- coding: utf-8 -*-
# # Visualize OH–FP variable/cluster correspondence from the three CSVs.

# import pandas as pd
# import numpy as np
# from pathlib import Path
# import matplotlib.pyplot as plt
# import matplotlib
# from matplotlib import font_manager as fm

# # ========= USER SETTINGS =========
# ROOT = Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap")
# MODE  = "cumeig"      # "top3" or "cumeig"
# INDEX = "dunn"        # "silhouette","dunn","gap","ch","db","ptbiserial"
# TOP_LABELS = 15       # 材料散布図で名前を表示する最大件数（purity高い順）

# # ========= Font (JP対応) =========
# def set_font():
#     cand = ["Noto Sans CJK JP","Noto Sans JP","Source Han Sans JP","IPAexGothic","Yu Gothic","Meiryo"]
#     have = set()
#     for p in fm.findSystemFonts(fontext="ttf"):
#         try: have.add(fm.FontProperties(fname=p).get_name())
#         except: pass
#     for w in cand:
#         if any(w.lower() in h.lower() for h in have):
#             matplotlib.rcParams["font.family"] = w
#             break
#     matplotlib.rcParams["axes.unicode_minus"] = False

# # ========= IO =========
# def load_tables(root: Path, mode: str, index: str):
#     f1 = root / f"material_to_fpcluster_{mode}_{index}.csv"
#     f2 = root / f"materialpair_cohesion_{mode}_{index}.csv"
#     f3 = root / f"ohcluster_consistency_{mode}_{index}.csv"
#     if not f1.exists():
#         raise FileNotFoundError(f"Not found: {f1}")
#     df_mat = pd.read_csv(f1)
#     df_pair = pd.read_csv(f2) if f2.exists() else pd.DataFrame()
#     df_ohc  = pd.read_csv(f3) if f3.exists() else pd.DataFrame()
#     return df_mat, df_pair, df_ohc

# # ========= PLOTS =========
# def plot_material_scatter(df_mat: pd.DataFrame, outdir: Path, mode: str, index: str):
#     set_font()
#     df = df_mat.copy()
#     # サイズ：出現サンプル数、色：OHクラスタ
#     size = 30 + 3 * df["n_samples_with_material"].fillna(0).astype(float)
#     c    = df["OH_cluster"].astype("category")

#     fig, ax = plt.subplots(figsize=(9,6), dpi=300)
#     sc = ax.scatter(df["FP_entropy"], df["FP_purity"], c=c.cat.codes, s=size, cmap="tab20", alpha=0.85, edgecolors="none")
#     ax.set_xlabel("FP entropy (larger = more fragmented)")
#     ax.set_ylabel("FP purity (larger = more cohesive)")
#     ax.set_title(f"Materials: Cohesion vs Fragmentation\n({mode} × {index})")

#     # ラベル（上位）
#     lab_df = df.sort_values(["FP_purity","n_samples_with_material"], ascending=False).head(TOP_LABELS)
#     for _, r in lab_df.iterrows():
#         ax.text(r["FP_entropy"], r["FP_purity"]+0.015, str(r["material"]), ha="center", va="bottom", fontsize=8)

#     # グリッド & 凡例
#     ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)
#     # OHクラスタ凡例（ユニークが多いと煩雑なので上限10）
#     uniq = sorted(df["OH_cluster"].dropna().unique())[:10]
#     handles = [plt.Line2D([0],[0], marker='o', color='w',
#                           markerfacecolor=plt.cm.tab20(i/20), markersize=8, label=f"OH {int(k)}")
#                for i,k in enumerate(uniq)]
#     if handles:
#         ax.legend(handles=handles, title="OH clusters (subset)", loc="upper right", frameon=False, fontsize=8)

#     outdir.mkdir(parents=True, exist_ok=True)
#     fig.tight_layout()
#     fig.savefig(outdir / "Fig_material_scatter_entropy_vs_purity.png", bbox_inches="tight")
#     fig.savefig(outdir / "Fig_material_scatter_entropy_vs_purity.pdf", bbox_inches="tight")
#     plt.close(fig)

# def plot_pair_distributions(df_pair: pd.DataFrame, outdir: Path, mode: str, index: str):
#     if df_pair is None or df_pair.empty:
#         return
#     set_font()
#     # 1) FP_major_same の割合（全体とOHクラスタ別：件数上位のみ）
#     fig, ax = plt.subplots(figsize=(8,4), dpi=300)
#     rate = df_pair["FP_major_same"].value_counts(normalize=True).rename({True:"Same", False:"Different"})
#     ax.bar(rate.index.astype(str), rate.values, color=["#4E79A7","#E15759"], alpha=0.9)
#     ax.set_ylim(0,1); ax.set_ylabel("Proportion"); ax.set_title(f"Pair-level: FP major same? ({mode} × {index})")
#     for xi, val in enumerate(rate.values):
#         ax.text(xi, val+0.02, f"{val:.2f}", ha="center", va="bottom")
#     fig.tight_layout()
#     fig.savefig(outdir / "Fig_pair_major_same_overall.png", bbox_inches="tight")
#     fig.savefig(outdir / "Fig_pair_major_same_overall.pdf", bbox_inches="tight")
#     plt.close(fig)

#     # 2) Cosine similarity と JS divergence の分布（ヒストグラム）
#     for col, xlabel in [("FP_cosine_similarity","Cosine similarity (1=similar)"),
#                         ("FP_JS_divergence","JS divergence (0=similar)")]:
#         s = df_pair[col].dropna()
#         if not len(s): 
#             continue
#         fig, ax = plt.subplots(figsize=(8,4), dpi=300)
#         ax.hist(s, bins=20, alpha=0.9)
#         ax.set_title(f"Pair-level distribution: {col} ({mode} × {index})")
#         ax.set_xlabel(xlabel); ax.set_ylabel("Count")
#         fig.tight_layout()
#         fig.savefig(outdir / f"Fig_pair_{col}_hist.png", bbox_inches="tight")
#         fig.savefig(outdir / f"Fig_pair_{col}_hist.pdf", bbox_inches="tight")
#         plt.close(fig)

# def plot_ohcluster_consistency(df_ohc: pd.DataFrame, outdir: Path, mode: str, index: str):
#     if df_ohc is None or df_ohc.empty:
#         return
#     set_font()
#     df = df_ohc.sort_values("cohesive_ratio", ascending=False).copy()
#     fig, ax = plt.subplots(figsize=(8,4), dpi=300)
#     x = np.arange(len(df))
#     ax.bar(x, df["cohesive_ratio"].values, color="#59A14F", alpha=0.9)
#     ax.set_xticks(x); ax.set_xticklabels([f"OH {int(c)}" for c in df["OH_cluster"]], rotation=35, ha="right")
#     ax.set_ylim(0,1.05)
#     ax.set_ylabel("Cohesive ratio (pair-level)")
#     ax.set_title(f"OH-cluster consistency in FP space ({mode} × {index})")
#     for xi, v in zip(x, df["cohesive_ratio"].values):
#         ax.text(xi, v+0.02, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
#     ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
#     fig.tight_layout()
#     fig.savefig(outdir / "Fig_ohcluster_consistency.png", bbox_inches="tight")
#     fig.savefig(outdir / "Fig_ohcluster_consistency.pdf", bbox_inches="tight")
#     plt.close(fig)

# def plot_mini_sankey(df_mat: pd.DataFrame, outdir: Path, mode: str, index: str, topN=20):
#     """材料→FP_major_cluster の対応をミニ・サンキーで可視化（上位N材料）。
#        matplotlibのSankeyは制約が強いので簡易版：横の帯（alluvial風）を描く。
#     """
#     if df_mat.empty:
#         return
#     set_font()
#     df = df_mat.sort_values("FP_purity", ascending=False).head(topN).copy()
#     df["OH_cluster"] = df["OH_cluster"].astype(int)
#     # 正規化幅
#     widths = (df["FP_purity"] + 0.2) / (df["FP_purity"].max() + 0.2)  # 0幅回避のためオフセット
#     y = 0.0
#     fig, ax = plt.subplots(figsize=(10, 0.35*len(df)+2), dpi=300)
#     for i, r in df.iterrows():
#         w = widths.loc[i]
#         # 左：材料名、右：FP_major_cluster
#         ax.plot([0, 1], [y, y], lw=8*w, alpha=0.9, color="#4E79A7")
#         ax.text(-0.02, y, str(r["material"]), ha="right", va="center", fontsize=8)
#         ax.text(1.02, y, f"FP {int(r['FP_major_cluster'])}", ha="left", va="center", fontsize=8, color="#555")
#         y += 0.5
#     ax.set_xlim(-0.25, 1.25); ax.set_ylim(-0.5, y-0.0)
#     ax.axis("off")
#     ax.set_title(f"Mini-alluvial: Material → FP major cluster (top {topN} by purity)\n({mode} × {index})", loc="left")
#     fig.tight_layout()
#     outdir.mkdir(parents=True, exist_ok=True)
#     fig.savefig(outdir / "Fig_material_to_FPmajor_miniAlluvial.png", bbox_inches="tight")
#     fig.savefig(outdir / "Fig_material_to_FPmajor_miniAlluvial.pdf", bbox_inches="tight")
#     plt.close(fig)

# # ========= main =========
# def main():
#     outdir = ROOT / f"figs_{MODE}_{INDEX}"
#     df_mat, df_pair, df_ohc = load_tables(ROOT, MODE, INDEX)

#     plot_material_scatter(df_mat, outdir, MODE, INDEX)     # 材料：purity vs entropy
#     plot_pair_distributions(df_pair, outdir, MODE, INDEX)  # ペア：major same割合、cosine/JS
#     plot_ohcluster_consistency(df_ohc, outdir, MODE, INDEX)# OHクラスタ：cohesive ratio
#     plot_mini_sankey(df_mat, outdir, MODE, INDEX, topN=20) # 材料→FPメジャー（簡易アリビアル）

#     print(f"[DONE] Saved figures under: {outdir}")

# if __name__ == "__main__":
#     main()

[DONE] Saved figures under: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap/figs_cumeig_dunn


In [3]:
# -*- coding: utf-8 -*-
"""
全 (mode,index) を For ループで一括可視化
- 事前に作成済みの 3種CSV（material_to_fpcluster / materialpair_cohesion / ohcluster_consistency）
  を読み、各条件ごとに図を保存
- 欠損はスキップ、最後に全条件のサマリーをまとめてCSV出力
"""

import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib
from matplotlib import font_manager as fm

# ========= ユーザー設定 =========
ROOT_VARMAP = Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap")
DIMS    = ["top3","cumeig"]
INDICES = ["silhouette","dunn","gap","ch","db","ptbiserial"]
TOP_LABELS = 15  # 材料散布図でラベル表示する材料数（purity高い順）

# ========= フォント（日本語可） =========
def set_font():
    cand = ["Noto Sans CJK JP","Noto Sans JP","Source Han Sans JP","IPAexGothic","Yu Gothic","Meiryo"]
    have = set()
    for p in fm.findSystemFonts(fontext="ttf"):
        try: have.add(fm.FontProperties(fname=p).get_name())
        except: pass
    for w in cand:
        if any(w.lower() in h.lower() for h in have):
            matplotlib.rcParams["font.family"] = w
            break
    matplotlib.rcParams["axes.unicode_minus"] = False

# ========= IO =========
def load_tables(root: Path, mode: str, index: str):
    f1 = root / f"material_to_fpcluster_{mode}_{index}.csv"
    f2 = root / f"materialpair_cohesion_{mode}_{index}.csv"
    f3 = root / f"ohcluster_consistency_{mode}_{index}.csv"
    if not f1.exists():
        return None, None, None
    df_mat = pd.read_csv(f1)
    df_pair = pd.read_csv(f2) if f2.exists() else pd.DataFrame()
    df_ohc  = pd.read_csv(f3) if f3.exists() else pd.DataFrame()
    return df_mat, df_pair, df_ohc

# ========= PLOTS =========
def plot_material_scatter(df_mat: pd.DataFrame, outdir: Path, mode: str, index: str, top_labels:int=15):
    set_font()
    df = df_mat.copy()
    if df.empty:
        return
    size = 30 + 3 * df["n_samples_with_material"].fillna(0).astype(float)
    c    = df["OH_cluster"].astype("category")

    fig, ax = plt.subplots(figsize=(9,6), dpi=300)
    sc = ax.scatter(df["FP_entropy"], df["FP_purity"], c=c.cat.codes, s=size, cmap="tab20", alpha=0.85, edgecolors="none")
    ax.set_xlabel("FP entropy (larger = more fragmented)")
    ax.set_ylabel("FP purity (larger = more cohesive)")
    ax.set_title(f"Materials: Cohesion vs Fragmentation ({mode} × {index})")

    lab_df = df.sort_values(["FP_purity","n_samples_with_material"], ascending=False).head(top_labels)
    for _, r in lab_df.iterrows():
        ax.text(r["FP_entropy"], r["FP_purity"]+0.015, str(r["material"]), ha="center", va="bottom", fontsize=8)

    ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)
    outdir.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(outdir / "Fig_material_scatter_entropy_vs_purity.png", bbox_inches="tight")
    fig.savefig(outdir / "Fig_material_scatter_entropy_vs_purity.pdf", bbox_inches="tight")
    plt.close(fig)

def plot_pair_distributions(df_pair: pd.DataFrame, outdir: Path, mode: str, index: str):
    if df_pair is None or df_pair.empty:
        return
    set_font()
    # 1) FP_major_same 割合（全体）
    fig, ax = plt.subplots(figsize=(8,4), dpi=300)
    rate = df_pair["FP_major_same"].value_counts(normalize=True).rename({True:"Same", False:"Different"})
    idx = ["Same","Different"]
    vals = [rate.get("Same",0.0), rate.get("Different",0.0)]
    ax.bar(idx, vals, color=["#4E79A7","#E15759"], alpha=0.9)
    ax.set_ylim(0,1); ax.set_ylabel("Proportion")
    ax.set_title(f"Pair-level: FP major same? ({mode} × {index})")
    for xi, val in enumerate(vals):
        ax.text(xi, val+0.02, f"{val:.2f}", ha="center", va="bottom")
    fig.tight_layout()
    fig.savefig(outdir / "Fig_pair_major_same_overall.png", bbox_inches="tight")
    fig.savefig(outdir / "Fig_pair_major_same_overall.pdf", bbox_inches="tight")
    plt.close(fig)

    # 2) Cosine similarity / JS divergence ヒスト
    for col, xlabel in [("FP_cosine_similarity","Cosine similarity (1=similar)"),
                        ("FP_JS_divergence","JS divergence (0=similar)")]:
        s = df_pair[col].dropna()
        if not len(s): 
            continue
        fig, ax = plt.subplots(figsize=(8,4), dpi=300)
        ax.hist(s, bins=20, alpha=0.9)
        ax.set_title(f"Pair-level distribution: {col} ({mode} × {index})")
        ax.set_xlabel(xlabel); ax.set_ylabel("Count")
        fig.tight_layout()
        fig.savefig(outdir / f"Fig_pair_{col}_hist.png", bbox_inches="tight")
        fig.savefig(outdir / f"Fig_pair_{col}_hist.pdf", bbox_inches="tight")
        plt.close(fig)

def plot_ohcluster_consistency(df_ohc: pd.DataFrame, outdir: Path, mode: str, index: str):
    if df_ohc is None or df_ohc.empty:
        return
    set_font()
    df = df_ohc.sort_values("cohesive_ratio", ascending=False).copy()
    fig, ax = plt.subplots(figsize=(8,4), dpi=300)
    x = np.arange(len(df))
    ax.bar(x, df["cohesive_ratio"].values, color="#59A14F", alpha=0.9)
    ax.set_xticks(x); ax.set_xticklabels([f"OH {int(c)}" for c in df["OH_cluster"]], rotation=35, ha="right")
    ax.set_ylim(0,1.05)
    ax.set_ylabel("Cohesive ratio (pair-level)")
    ax.set_title(f"OH-cluster consistency in FP space ({mode} × {index})")
    for xi, v in zip(x, df["cohesive_ratio"].values):
        ax.text(xi, v+0.02, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
    ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
    fig.tight_layout()
    fig.savefig(outdir / "Fig_ohcluster_consistency.png", bbox_inches="tight")
    fig.savefig(outdir / "Fig_ohcluster_consistency.pdf", bbox_inches="tight")
    plt.close(fig)

def plot_mini_sankey(df_mat: pd.DataFrame, outdir: Path, mode: str, index: str, topN=20):
    if df_mat is None or df_mat.empty:
        return
    set_font()
    df = df_mat.sort_values("FP_purity", ascending=False).head(topN).copy()
    if df.empty:
        return
    # 幅正規化（0回避のオフセット）
    widths = (df["FP_purity"] + 0.2) / (df["FP_purity"].max() + 0.2)
    y = 0.0
    fig, ax = plt.subplots(figsize=(10, 0.35*len(df)+2), dpi=300)
    for _, r in df.iterrows():
        w = widths.loc[_]
        ax.plot([0, 1], [y, y], lw=8*w, alpha=0.9, color="#4E79A7")
        ax.text(-0.02, y, str(r["material"]), ha="right", va="center", fontsize=8)
        # FP_major_cluster がNaNのときは表示スキップ
        fp_major = r.get("FP_major_cluster")
        right_lbl = f"FP {int(fp_major)}" if pd.notna(fp_major) else "FP –"
        ax.text(1.02, y, right_lbl, ha="left", va="center", fontsize=8, color="#555")
        y += 0.5
    ax.set_xlim(-0.25, 1.25); ax.set_ylim(-0.5, y-0.0)
    ax.axis("off")
    ax.set_title(f"Mini-alluvial: Material → FP major cluster (top {topN} by purity)\n({mode} × {index})", loc="left")
    fig.tight_layout()
    outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / "Fig_material_to_FPmajor_miniAlluvial.png", bbox_inches="tight")
    fig.savefig(outdir / "Fig_material_to_FPmajor_miniAlluvial.pdf", bbox_inches="tight")
    plt.close(fig)

# ========= メイン：Forループで総なめ =========
def main():
    summary_rows = []  # 全条件の要約（後でCSVに）
    for mode in DIMS:
        for index in INDICES:
            df_mat, df_pair, df_ohc = load_tables(ROOT_VARMAP, mode, index)
            if df_mat is None:  # material_to_fpcluster が無い条件はスキップ
                print(f"[SKIP] No data for {mode} × {index}")
                continue

            outdir = ROOT_VARMAP / f"figs_{mode}_{index}"
            outdir.mkdir(parents=True, exist_ok=True)

            # 可視化
            plot_material_scatter(df_mat, outdir, mode, index, TOP_LABELS)
            plot_pair_distributions(df_pair, outdir, mode, index)
            plot_ohcluster_consistency(df_ohc, outdir, mode, index)
            plot_mini_sankey(df_mat, outdir, mode, index, topN=20)
            print(f"[DONE] Saved figs: {mode} × {index} -> {outdir}")

            # 要約（条件ごと）
            # 代表統計：材料数、平均purity、平均entropy、major_same率、平均cosine、平均JSD、OHクラスタcohesiveの中央値
            n_mat = len(df_mat)
            mean_purity = df_mat["FP_purity"].dropna().mean() if "FP_purity" in df_mat else np.nan
            mean_entropy = df_mat["FP_entropy"].dropna().mean() if "FP_entropy" in df_mat else np.nan
            major_same_rate = df_pair["FP_major_same"].mean() if (df_pair is not None and not df_pair.empty) else np.nan
            mean_cosine = df_pair["FP_cosine_similarity"].dropna().mean() if (df_pair is not None and not df_pair.empty) else np.nan
            mean_jsd = df_pair["FP_JS_divergence"].dropna().mean() if (df_pair is not None and not df_pair.empty) else np.nan
            median_cohesive = df_ohc["cohesive_ratio"].median() if (df_ohc is not None and not df_ohc.empty) else np.nan

            summary_rows.append({
                "mode": mode, "index": index,
                "n_materials": n_mat,
                "mean_FP_purity": mean_purity,
                "mean_FP_entropy": mean_entropy,
                "pair_FP_major_same_rate": major_same_rate,
                "pair_mean_cosine_similarity": mean_cosine,
                "pair_mean_JS_divergence": mean_jsd,
                "OHcluster_median_cohesive_ratio": median_cohesive
            })

    if summary_rows:
        df_sum = pd.DataFrame(summary_rows).sort_values(["mode","index"])
        outsum = ROOT_VARMAP / "summary_all_conditions.csv"
        df_sum.to_csv(outsum, index=False)
        print(f"[SAVED] Summary across all conditions: {outsum}")
    else:
        print("[INFO] No figures generated. Check CSV availability under OHFP_varmap/")

if __name__ == "__main__":
    main()

[DONE] Saved figs: top3 × silhouette -> /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap/figs_top3_silhouette
[DONE] Saved figs: top3 × dunn -> /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap/figs_top3_dunn
[DONE] Saved figs: top3 × gap -> /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap/figs_top3_gap
[DONE] Saved figs: top3 × ch -> /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap/figs_top3_ch
[DONE] Saved figs: top3 × db -> /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap/figs_top3_db
[DONE] Saved figs: top3 × ptbiserial -> /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap/figs_top3_ptbiserial
[DONE] Saved figs: cumeig × silhouette -> /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap/figs_cumeig_silhouette
[DONE] Saved figs: cumeig × dunn -> /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap/figs_cumeig_dun